# Problem 1: Basic Inspection and Cleaning of Raw REFIT Data (House 1)

**Objective:**
In this notebook, we perform end-to-end data inspection, cleaning, and 1-minute resampling of raw electrical load data for **House 1** from the REFIT dataset.

### Key Steps Covered:
1. **Raw Data Merging:** Combine Part 1 and Part 2 (switching at `2014-06-27` to utilize explicit `NaN` tracking for unavailable sensors).
2. **Data Quality Audit:** Check timestamp ordering, quantify duplicate timestamps, analyze sampling intervals, and inspect extreme/impossible power values.
3. **Cleaning Pipeline:**
   - Chronological timestamp sorting.
   - Deduplication (keeping the last observation).
   - Clamping physical impossibilities (Aggregate > 15 kW, Appliances > 4 kW, negatives).
   - Uniform 1-minute resampling (`mean` power).
   - Handling gaps: short forward-fill (<= 5 mins) vs. explicit outage flagging (> 5 mins).
4. **Visual Verification:** Before vs. after cleaning plots for Aggregate power, Appliance 1 (Fridge), and Appliance 5 (Washing Machine), plus spike and outage handling plots.

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)
print('Libraries successfully imported!')

## Step 1: Load Raw REFIT Data (Part 1 and Part 2)
The raw data for House 1 is split into two files:
- `RAW_House1_Part1.csv`: Covers the first period where missing/unavailable sensor readings were recorded as `0`.
- `RAW_House1_Part2.csv`: Covers the second period (starting `2014-06-27 21:57:50`) where unavailable sensors were explicitly logged as `NaN`.
We use `float32` for power readings to optimize memory efficiency.

In [ ]:
PART1_PATH = 'RAW_House1_Part1.csv'
PART2_PATH = 'RAW_House1_Part2.csv'

channels = ['Aggregate'] + [f'Appliance{i}' for i in range(1, 10)]
dtype_dict = {col: 'float32' for col in channels}

print('Loading Part 1...')
df1 = pd.read_csv(PART1_PATH, dtype=dtype_dict)
print(f"  Part 1 rows: {len(df1):,}, range: {df1['Time'].iloc[0]} to {df1['Time'].iloc[-1]}")

print('Loading Part 2...')
df2 = pd.read_csv(PART2_PATH, dtype=dtype_dict)
len_df2 = len(df2)
print(f"  Part 2 rows: {len_df2:,}, range: {df2['Time'].iloc[0]} to {df2['Time'].iloc[-1]}")

# Switch to Part 2 where explicit NaNs begin
SWITCH_TIME = '2014-06-27 21:57:50'
df1 = df1[df1['Time'] < SWITCH_TIME]
len_df1_cut = len(df1)

raw_combined = pd.concat([df1, df2], ignore_index=True)
len_combined = len(raw_combined)
del df1, df2
gc.collect()

print(f'Total combined raw records: {len_combined:,}')

## Step 2: Raw Data Quality Inspection
Here we audit the 5 core quality aspects required by the assignment:
1. **Timestamp parsing and chronological ordering**
2. **Duplicate timestamps**
3. **Sampling intervals (irregular intervals & outages)**
4. **Negative values and extreme physical spikes**

In [ ]:
# 1. Parse timestamps and check ordering
raw_combined['DateTime'] = pd.to_datetime(raw_combined['Time'])
raw_combined.drop(columns=['Time', 'Unix'], inplace=True)
is_monotonic = raw_combined['DateTime'].is_monotonic_increasing
print(f'1. Timestamps strictly ordered initially? {is_monotonic}')

# 2. Duplicate timestamps
duplicate_count = raw_combined.duplicated(subset=['DateTime']).sum()
print(f'2. Duplicate timestamps found: {duplicate_count:,} ({duplicate_count / len(raw_combined):.2%})')

# Sort chronologically to inspect intervals
raw_combined.sort_values('DateTime', inplace=True)
raw_combined.reset_index(drop=True, inplace=True)

# 3. Sampling intervals
intervals = raw_combined['DateTime'].diff().dt.total_seconds().dropna()
median_interval = intervals.median()
p25_interval = intervals.quantile(0.25)
p75_interval = intervals.quantile(0.75)
max_interval_hours = intervals.max() / 3600.0

print('3. Sampling interval statistics (seconds):')
print(f'   Median interval: {median_interval:.1f} s')
print(f'   25th percentile: {p25_interval:.1f} s')
print(f'   75th percentile: {p75_interval:.1f} s')
print(f'   Max interval (outage duration): {max_interval_hours:.1f} hours')
del intervals
gc.collect()

# 4. Value sanity checks
negative_readings = (raw_combined[channels] < 0).sum().sum()
aggregate_spikes = (raw_combined['Aggregate'] > 15000).sum()
appliance_spikes = sum((raw_combined[f'Appliance{i}'] > 4000).sum() for i in range(1, 10))

print('4. Value sanity checks:')
print(f'   Negative power values: {negative_readings}')
print(f'   Aggregate spikes (> 15,000 W): {aggregate_spikes:,}')
print(f'   Appliance spikes (> 4,000 W): {appliance_spikes:,}')

## Step 3: Documented Cleaning & 1-Minute Resampling Pipeline
We apply the following stated rules:
- **Deduplication Rule:** Keep `last` observation for identical timestamps.
- **Spike Clamping Rule:** Domestic single-phase UK circuits cannot sustain appliance loads above 4 kW (13A @ 230V = ~3 kW max rating). Extreme spikes (>4 kW on IAMs and >15 kW on Aggregate) are sensor artifacts and are replaced with `NaN`.
- **Resampling Rule:** Aggregate readings to uniform 1-minute bins using the interval `mean`.
- **Short Gaps vs Outage Rule:** Gaps <= 5 minutes are forward-filled (minor network jitter). Longer outages (> 5 minutes) are kept as `NaN` and flagged with `outage_flag = 1`.

In [ ]:
# Cache a 48h slice for raw before/after plotting
START_TIME = '2013-10-22 00:00:00'
END_TIME = '2013-10-24 00:00:00'
raw_slice = raw_combined[(raw_combined['DateTime'] >= START_TIME) & (raw_combined['DateTime'] <= END_TIME)].copy()

SPIKE_START = '2014-01-10'
SPIKE_END = '2014-01-12'
spike_window_raw = raw_combined[(raw_combined['DateTime'] >= SPIKE_START) & (raw_combined['DateTime'] <= SPIKE_END)].copy()

# 1. Deduplicate
raw_combined.drop_duplicates(subset=['DateTime'], keep='last', inplace=True)

# 2. Clamping spikes
raw_combined.loc[raw_combined['Aggregate'] > 15000, 'Aggregate'] = np.nan
raw_combined.loc[raw_combined['Aggregate'] < 0, 'Aggregate'] = np.nan
for i in range(1, 10):
    col = f'Appliance{i}'
    raw_combined.loc[raw_combined[col] > 4000, col] = np.nan
    raw_combined.loc[raw_combined[col] < 0, col] = np.nan

# 3. Resample to 1 minute
raw_combined.set_index('DateTime', inplace=True)
df_1min = raw_combined[channels].resample('1min').mean()
del raw_combined
gc.collect()

# 4. Fill short gaps & flag long outages
df_1min_filled = df_1min.ffill(limit=5)
del df_1min
gc.collect()

df_1min_filled['outage_flag'] = df_1min_filled['Aggregate'].isna().astype(int)
outage_pct = df_1min_filled['outage_flag'].mean() * 100
print(f"Cleaning complete! Outage minutes flagged: {df_1min_filled['outage_flag'].sum():,} ({outage_pct:.2f}%)")

## Step 4: Visual Verification Before and After Cleaning
We plot:
1. **Aggregate Power, Appliance 1 (Fridge), and Appliance 5 (Washing Machine)** across a representative 48-hour window.
2. **Demonstration of Spike Removal and Long Outage Flagging**.

In [ ]:
clean_slice = df_1min_filled.loc[START_TIME:END_TIME].copy()

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex='col', sharey='row')

# Aggregate
axes[0, 0].plot(raw_slice['DateTime'], raw_slice['Aggregate'], color='#1f77b4', alpha=0.7, lw=0.8)
axes[0, 0].set_title('Raw: Aggregate Power (House 1 - 8s interval)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Power (Watts)')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

axes[0, 1].plot(clean_slice.index, clean_slice['Aggregate'], color='#2ca02c', lw=1.2)
axes[0, 1].set_title('Cleaned: Aggregate Power (1-minute Resampled)', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# Appliance 1 (Fridge)
axes[1, 0].plot(raw_slice['DateTime'], raw_slice['Appliance1'], color='#ff7f0e', alpha=0.7, lw=0.8)
axes[1, 0].set_title('Raw: Appliance 1 (Fridge - noisy/jittered)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Power (Watts)')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

axes[1, 1].plot(clean_slice.index, clean_slice['Appliance1'], color='#d62728', lw=1.2)
axes[1, 1].set_title('Cleaned: Appliance 1 (Fridge - 1-minute Mean)', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)

# Appliance 5 (Washing Machine)
axes[2, 0].plot(raw_slice['DateTime'], raw_slice['Appliance5'], color='#9467bd', alpha=0.7, lw=0.8)
axes[2, 0].set_title('Raw: Appliance 5 (Washing Machine - Raw Readings)', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Power (Watts)')
axes[2, 0].set_xlabel('Timestamp')
axes[2, 0].grid(True, linestyle='--', alpha=0.5)

axes[2, 1].plot(clean_slice.index, clean_slice['Appliance5'], color='#8c564b', lw=1.2)
axes[2, 1].set_title('Cleaned: Appliance 5 (Washing Machine - 1-minute Resampled)', fontsize=11, fontweight='bold')
axes[2, 1].set_ylabel('Power (Watts)')
axes[2, 1].set_xlabel('Timestamp')
axes[2, 1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('House 1: Raw vs Cleaned Power Comparison (48-Hour Representative Window)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/problem1_before_after_cleaning.png', dpi=200)
plt.show()

In [ ]:
# Plot spike removal and outage handling
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
spike_window_clean = df_1min_filled.loc[SPIKE_START:SPIKE_END]

ax1.plot(spike_window_raw['DateTime'], spike_window_raw['Aggregate'], label='Raw Aggregate (with sensor spikes)', color='orange', alpha=0.8)
ax1.plot(spike_window_clean.index, spike_window_clean['Aggregate'], label='Cleaned Aggregate (spikes removed & resampled)', color='navy', lw=1.2)
ax1.set_title('Demonstration of Sensor Spike Removal & Signal Smoothing', fontsize=11, fontweight='bold')
ax1.set_ylabel('Power (Watts)')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

# Outage flagging demonstration (February 2014 REFIT downtime)
feb_outage = df_1min_filled.loc['2014-02-18':'2014-02-25']
ax2.plot(feb_outage.index, feb_outage['Aggregate'], label='Cleaned Aggregate Power', color='teal', lw=1.2)
ax2.fill_between(feb_outage.index, 0, 3000, where=(feb_outage['outage_flag'] == 1),
                 color='crimson', alpha=0.25, label='Flagged Long Outage (Gaps > 5 mins left as NaN)')
ax2.set_title('Demonstration of Outage Flagging (February 2014 REFIT Downtime)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Power (Watts)')
ax2.set_xlabel('Timestamp')
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/problem1_outage_spike_handling.png', dpi=200)
plt.show()

## Step 5: Save Cleaned 1-Minute Dataset
We save the cleaned dataset to `house1_cleaned_1min.parquet` (compact and fast) and export a sample to CSV.

In [ ]:
OUTPUT_DATA_PATH = 'house1_cleaned_1min.parquet'
df_1min_filled.to_parquet(OUTPUT_DATA_PATH)
print(f'Saved cleaned dataset to: {OUTPUT_DATA_PATH}')

# Also export sample CSV for inspection
df_1min_filled.head(10000).to_csv('house1_cleaned_1min_sample.csv')
print('Exported sample CSV: house1_cleaned_1min_sample.csv')